makes yields from the signal and background. Writes to a new file.

In [1]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array
import vector

In [2]:
# opening the file
# CHECK LOCATION FROM LAST CELL OF merge_hists.ipynb

#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch/merged_all_batches.root"
#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/test_merged_all_batches.root"
#filepath = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/per_batch/merged_all_batches.root"

#signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged/merged_signals.root"
#background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [3]:
signal_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/Merged_Signals.root"
background_path = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/Merged_Backgrounds.root"

ROOT.gDirectory.Clear()

In [4]:
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [5]:
def makeYields(signal_path, background_path):
    '''
    takes a filepath and makes the yields
    returns:
    - yields per mult bin
    '''

    sig_file = ROOT.TFile.Open(signal_path, "READ")
    bkg_file = ROOT.TFile.Open(background_path, "READ")

    wta_yields = {}
    std_yields = {}
    y_axis_title = "#frac{1}{N_{trig}} #frac{d^{2}N^{pair}}{d#Delta#phi*#eta*}"

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

        # open histograms
        hSig_wta = sig_file.Get(f"WTA_sig_{mult_bin[0]}_{mult_bin[1]}")
        hSig_std = sig_file.Get(f"STD_sig_{mult_bin[0]}_{mult_bin[1]}")
        
        hBkg_wta = bkg_file.Get(f"WTA_bkg_{mult_bin[0]}_{mult_bin[1]}")
        hBkg_std = bkg_file.Get(f"STD_bkg_{mult_bin[0]}_{mult_bin[1]}")

        # get num jets
        num_jets_wta = sig_file.Get(f"num_jets_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        num_jets_std = sig_file.Get(f"num_jets_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        # clone signals
        h_yield_wta = hSig_wta.Clone(f"WTA_yield_{mult_bin[0]}_{mult_bin[1]}")
        h_yield_std = hSig_std.Clone(f"STD_yield_{mult_bin[0]}_{mult_bin[1]}")

        # detach
        h_yield_wta.SetDirectory(0)
        h_yield_std.SetDirectory(0)
        
        # name axes
        h_yield_wta.SetTitle(f"WTA Yield ({bin_key});#Delta#eta*;#Delta#phi*;{y_axis_title}")
        h_yield_std.SetTitle(f"Standard Yield ({bin_key});#Delta#eta*;#Delta#phi*;{y_axis_title}")


        # find B(0,0)
        bin_zero_wta = hBkg_wta.FindBin(0.0, 0.0)
        bin_zero_std = hBkg_std.FindBin(0.0, 0.0)

        b00_wta = hBkg_wta.GetBinContent(bin_zero_wta)
        b00_std = hBkg_std.GetBinContent(bin_zero_std)

        # signal / background
        h_yield_wta.Divide(hBkg_wta)
        h_yield_std.Divide(hBkg_std)

        h_yield_wta.Scale(b00_wta / num_jets_wta)
        h_yield_std.Scale(b00_std / num_jets_std)

        wta_yields[bin_key] = h_yield_wta
        std_yields[bin_key] = h_yield_std

        print(f"{bin_key}")
        print(f"wta b00: {b00_wta}")
        print(f"std b00: {b00_std}")
        print(f"wta num_jets: {num_jets_wta}")
        print(f"std num_jets: {num_jets_std}")

    sig_file.Close()
    bkg_file.Close()
    
    return wta_yields, std_yields

In [6]:
def get_avg_Nch(signal_path):
    '''
    does what it says on the box
    '''
    wta_avg_Nch_dict = {}
    std_avg_Nch_dict = {}

    file = ROOT.TFile.Open(signal_path, "READ")

    for mult_bin in analysis_bins:
        bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"
        # read avg Nch parameters
        avg_Nch_wta = file.Get(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}").GetVal()
        avg_Nch_std = file.Get(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}").GetVal()

        wta_avg_Nch_dict[bin_key] = avg_Nch_wta
        std_avg_Nch_dict[bin_key] = avg_Nch_std

    return wta_avg_Nch_dict, std_avg_Nch_dict


In [7]:
# make yields, get avg Nch values

wta_yields, std_yields = makeYields(signal_path, background_path)
wta_avg_Nch_dict, std_avg_Nch_dict = get_avg_Nch(signal_path)

0 <  Nch < 25
wta b00: 218542.0
std b00: 79784.0
wta num_jets: 28866.0
std num_jets: 10270.0
25 <  Nch < 36
wta b00: 8202376.0
std b00: 2183368.0
wta num_jets: 407849.0
std num_jets: 94552.0
36 <  Nch < 48
wta b00: 76099860.0
std b00: 14954060.0
wta num_jets: 1843920.0
std num_jets: 294627.0
48 <  Nch < 60
wta b00: 1015555904.0
std b00: 474805370.0
wta num_jets: 13689638.0
std num_jets: 4821626.0
60 <  Nch < 71
wta b00: 2518953192.0
std b00: 4167296912.0
wta num_jets: 26114955.0
std num_jets: 34516611.0
71 <  Nch < 78
wta b00: 421197696.0
std b00: 799349596.0
wta num_jets: 3057982.0
std num_jets: 4638779.0
78 <  Nch < 91
wta b00: 175339436.0
std b00: 372897688.0
wta num_jets: 989422.0
std num_jets: 1677858.0
91 <  Nch < 97
wta b00: 13413750.0
std b00: 32300552.0
wta num_jets: 56399.0
std num_jets: 108599.0
97 <  Nch < 1000
wta b00: 6963754.0
std b00: 18080336.0
wta num_jets: 24231.0
std num_jets: 50340.0


In [9]:
# save output

#EDIT FILE PATH IF NECESSARY

out_file = ROOT.TFile("/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/merged_yields.root", "RECREATE")

for mult_bin in analysis_bins:
    bin_key = f"{mult_bin[0]} <  Nch < {mult_bin[1]}"

    # write histograms
    wta_yields[bin_key].Write()
    std_yields[bin_key].Write()

    # write TParams
    param_avg_Nch_wta = ROOT.TParameter('double')(f"avg_Nch_WTA_{mult_bin[0]}_{mult_bin[1]}", wta_avg_Nch_dict[bin_key])
    param_avg_Nch_std = ROOT.TParameter('double')(f"avg_Nch_STD_{mult_bin[0]}_{mult_bin[1]}", std_avg_Nch_dict[bin_key])
    param_avg_Nch_wta.Write()
    param_avg_Nch_std.Write()

out_file.Close()